In [0]:
# Databricks notebook source

from datetime import datetime, timezone
import random
import uuid

statuses = ["CREATED", "PAID", "SHIPPED", "CANCELLED"]

events = []

for _ in range(100):
    event_type = random.choice(["INSERT", "UPDATE", "CANCEL"])

    events.append(
        {
            "event_id": str(uuid.uuid4()),
            "order_id": random.randint(10000, 99999),
            "customer_id": random.randint(1000, 9999),
            "event_type": event_type,
            "status": (
                "CANCELLED"
                if event_type == "CANCEL"
                else random.choice(statuses)
            ),
            "amount": round(random.uniform(5, 1000), 2),
            "sequence_number": random.randint(1, 10),
            "event_time": datetime.now(timezone.utc).isoformat()
        }
    )

events_df = spark.createDataFrame(events)

output_path = (
    "/Volumes/fintech_lakehouse/bronze/raw_events/"
    f"orders/batch_{datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S')}"
)

events_df.coalesce(1).write.mode("append").json(output_path)

print(f"Generated {len(events)} CDC events")
print(f"Written to: {output_path}")

display(events_df)

display(
    dbutils.fs.ls(
        "/Volumes/fintech_lakehouse/bronze/raw_events/orders/"
    )
)